# Library 指定兩句重錄｜每句三候選
僅Rosie兩句，原因：音色太成熟。維持qwen_rosie_r2核准原參考，不變調。28句已接受不推論，Princess af_bella不變。免費Colab請選T4 GPU；免費資源可能有限，無付費API。

## 1 安裝（新執行階段需要）

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 2 上傳 library-retakes2-inputs.zip

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile,io,sys,uuid
up=files.upload()
if len(up)!=1:raise ValueError('請只上傳一個輸入ZIP')
WORK=Path('/content/library_retakes2_'+uuid.uuid4().hex[:8])
with zipfile.ZipFile(io.BytesIO(next(iter(up.values())))) as z:
 if len(z.infolist())>300 or sum(i.file_size for i in z.infolist())>50000000:raise ValueError('ZIP異常')
 for n in z.namelist():
  if '\\' in n or ':' in n or not (WORK/n).resolve().is_relative_to(WORK.resolve()):raise ValueError('Unsafe ZIP')
 z.extractall(WORK)
sys.path.insert(0,str(WORK))
import retake_two as rt
from IPython.display import display,Audio,Markdown
from google.colab import output
output.enable_custom_widget_manager()

## 3 Dry-run：確認只有兩句，28段hash核對

In [ ]:
dry=rt.generate(WORK,Path('/content/library_retakes2_candidates'),run=False)
for a in dry['jobs']:print(a['lineId'],'\n',a['text'],'\n原因：',a['reason'])
print('2句 / 6候選 / generated=0；28段已接受SHA256通過。')

## 4 生成兩句×三候選
核對上方清單後，把RUN_RETAKES改True，CONFIRMATION填CONFIRM_LIBRARY_TWO_RETAKES。只改這格，不重生其他28段。

In [ ]:
RUN_RETAKES=False
CONFIRMATION=''
if RUN_RETAKES:
 BATCH=rt.generate(WORK,Path('/content/library_retakes2_candidates'),run=True,confirmation=CONFIRMATION)
 print('六候選另存：',BATCH)
else:print('尚未生成。請先核對兩句清單。')

## 5 比較原音及候選1／2／3，兩句都選好才匯出
先聽核准參考，判斷自然童真與活潑感，並檢查漏字、發音及多餘聲音。不同seed不保證更童真，以你的試聽為準。

In [ ]:
import ipywidgets as widgets
import shutil
if 'BATCH' not in globals():
 print('先執行第4區塊，或於下次恢復解壓的候選資料夾後設定BATCH。')
else:
 BATCH=Path(BATCH);cm=rt.base.read(BATCH/'candidate-manifest.json')
 m,review,lock,jobs=rt.verify(WORK)
 ref=m['characters']['rosie']['reference']
 print('核准Rosie參考：',ref['ref_text']);display(Audio(filename=str(WORK/'references'/ref['file'])))
 choice_file=BATCH/'candidate-selections.json'
 choices=rt.base.read(choice_file) if choice_file.exists() else {}
 selectors={}
 for a in jobs:
  print('\n',a['lineId'],'\n',a['text'],'\n原音（需要重錄）')
  display(Audio(filename=str(WORK/'originals'/a['src'])))
  for c in cm['candidates'][a['lineId']]:
   if rt.base.sha(BATCH/c['src'])!=c['sha256']:raise ValueError('候選音檔hash不符')
   print('候選',c['candidate']);display(Audio(filename=str(BATCH/c['src'])))
  selector=widgets.Dropdown(options=[('尚未選定',None),('候選1',1),('候選2',2),('候選3',3)],value=choices.get(a['lineId']),description='正式選擇：')
  selectors[a['lineId']]=selector;display(selector)
 def save_choice(_):
  rt.base.save(choice_file,{lid:c.value for lid,c in selectors.items()})
 for selector in selectors.values():selector.observe(save_choice,names='value')
 button=widgets.Button(description='選定後下載完整30段修正版',layout=widgets.Layout(width='280px'))
 backup=widgets.Button(description='下載六候選備份',layout=widgets.Layout(width='220px'))
 status=widgets.Output()
 def export_selected(_):
  with status:
   try:
    selected={lid:c.value for lid,c in selectors.items()}
    dest=Path('/content/library30_selected_'+uuid.uuid4().hex[:8])
    folder,archive=rt.export_selected(WORK,BATCH,selected,dest)
    print('28段SHA256不變；兩句採選定版本。下載：',archive)
    files.download(str(archive))
   except Exception as e:print('尚未匯出：',e)
 def export_backup(_):
  save_choice(None)
  files.download(shutil.make_archive('/content/library-two-candidates-'+uuid.uuid4().hex[:8],'zip',BATCH))
 button.on_click(export_selected);backup.on_click(export_backup);display(button,backup,status)

## 授權及限制
Qwen固定模型revision，Apache-2.0；核准參考音僅依家長提供用途，來源權利未另行獨立稽核，私人輸入ZIP勿公開。不使用其他真人聲線。此次不生成Princess/Bakery/Park，不修改正式遊戲。候選另存，選定前原30段唯讀；輸出仍待遊戲內驗收。Colab重置會清空未下載候選，離開前下載備份及修正版。